In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nusratjahannawal/banglapsg-v2-deduplicated/BanglaPSG_V2_Deduplicated.xlsx


In [2]:
!pip install -q transformers datasets accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 839.5 kB/s eta 0:00:00a 0:00:01


In [3]:
import pandas as pd
import numpy as np
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

In [4]:
import glob
import pandas as pd

hits = glob.glob("/kaggle/input/datasets/nusratjahannawal/banglapsg-v2-deduplicated/BanglaPSG_V2_Deduplicated.xlsx", recursive=True)
assert len(hits) == 1, f"Expected exactly one BanglaPSG_V2_Deduplicated.xlsx, found: {hits}"
path = hits[0]
print("Using:", path)

df = pd.read_excel(path)
assert df.columns.tolist() == ["No.", "Grievance Text", "Category", "Severity"], df.columns.tolist()
assert len(df) == 5970, len(df)
assert df["Grievance Text"].duplicated().sum() == 0, "Exact duplicate texts found: wrong file?"
df.head()

Using: /kaggle/input/datasets/nusratjahannawal/banglapsg-v2-deduplicated/BanglaPSG_V2_Deduplicated.xlsx


,No.,Grievance Text,Category,Severity
0,1,সরকারি হাসপাতাল গুলো এত অপরিষ্কার আর নোংরা।,Healthcare,High
1,2,মেডিসিন বিশেষজ্ঞ ডাঃ বসে না এসিস্ট্যান্ট দিয়ে...,Healthcare,High
2,3,রূপগঞ্জে মানুষ মারার কারখানা ইউ এস-বাংলা হাসপাতাল,Healthcare,Low
3,4,সরকারি হাসপাতালগুলোর সমস্ত টেস্টের মেশিন গুলো ...,Healthcare,Medium
4,5,সামান্য জ্বর আসলে এক হাজার নয়শত টাকার টেস্ট দয়,Healthcare,Low


In [5]:
data = df[["Grievance Text", "Severity"]].copy()

data.columns = ["text", "label"]

label_encoder = LabelEncoder()

data["label"] = label_encoder.fit_transform(data["label"])

print(label_encoder.classes_)

num_labels = len(label_encoder.classes_)

print("Number of classes:", num_labels)

['High' 'Low' 'Medium']
Number of classes: 3


In [6]:
#Leakage-safe split built from BanglaPSG_V2_Deduplicated.xlsx
import re, unicodedata, hashlib
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

_SEED = 42
_CHAR_COS, _WORD_COS, _CONTAIN, _MIN_WORDS = 0.65, 0.60, 0.90, 4
_RATIO = np.array([0.70, 0.15, 0.15])         # train / val / test

def _norm_key(t):
    t = unicodedata.normalize("NFC", str(t))
    t = re.sub(r"[\u200b\u200c\u200d\u2060\ufeff]", "", t)
    t = t.replace("\\n", " ")
    t = re.sub(r"[^\u0980-\u09FFA-Za-z0-9]+", " ", t)
    return re.sub(r"\s+", " ", t).strip().lower()

_text = df["Grievance Text"].astype(str).map(lambda t: unicodedata.normalize("NFC", t))
_ks = _text.map(_norm_key)
_key = _ks.str.replace(" ", "", regex=False)
_strata = (df["Category"].astype(str) + " | " + df["Severity"].astype(str)).values
_n = len(df)
assert _text.duplicated().sum() == 0 and _key.duplicated().sum() == 0, "Input still has duplicates"

# 1)link near-duplicates
def _pairs(X, thr):
    out = []
    for s in range(0, _n, 500):
        sim = (X[s:s + 500] @ X.T).toarray()
        r, c = np.where(sim >= thr)
        out += [(s + a, b) for a, b in zip(r, c) if b > s + a]
    return out

_Xc = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True).fit_transform(_ks)
_Xw = TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), sublinear_tf=True).fit_transform(_ks)
_links = set(_pairs(_Xc, _CHAR_COS)) | set(_pairs(_Xw, _WORD_COS))

_B = CountVectorizer(token_pattern=r"\S+", binary=True).fit_transform(_ks).tocsr().astype(np.float32)
_nw = np.asarray(_B.sum(axis=1)).ravel()
_inter = (_B @ _B.T).tocoo()
_m = _inter.row < _inter.col
_I, _J, _V = _inter.row[_m], _inter.col[_m], _inter.data[_m]
_mn = np.minimum(_nw[_I], _nw[_J])
_sel = (_mn >= _MIN_WORDS) & (_V / _mn >= _CONTAIN - 1e-9)
_links |= set(zip(_I[_sel].tolist(), _J[_sel].tolist()))

_L = np.array(sorted(_links))
_adj = coo_matrix((np.ones(len(_L)), (_L[:, 0], _L[:, 1])), shape=(_n, _n))
_, _grp = connected_components(_adj, directed=False)
assert np.bincount(_grp).max() <= 10, "A giant linked group formed"

# 2)stratified group assignment: pure numpy + hashing
_sid = np.unique(_strata, return_inverse=True)[1]
_S, _G = _sid.max() + 1, _grp.max() + 1
_comp = np.zeros((_G, _S))
np.add.at(_comp, (_grp, _sid), 1)                       
_first = {}
for _i, _g in enumerate(_grp):
    _first.setdefault(_g, _key.iat[_i])
_tie = np.array([int(hashlib.md5(f"{_SEED}|{_first[_g]}".encode()).hexdigest()[:12], 16) for _g in range(_G)])
_order = np.lexsort((_tie, -_comp.sum(1)))              
_tgt = _RATIO[:, None] * _comp.sum(0)[None, :]          
_cur = np.zeros((3, _S))
_assign = np.zeros(_G, dtype=int)
for _g in _order:
    _new = _cur + _comp[_g]
    _cost = (((_new - _tgt) ** 2 - (_cur - _tgt) ** 2) / np.maximum(_tgt, 1)).sum(axis=1)
    _s = int(np.argmin(_cost))
    _assign[_g] = _s
    _cur[_s] = _new[_s]
_split = np.array(["train", "val", "test"])[_assign[_grp]]
df["Group"], df["Split"] = _grp, _split
_dev = np.abs(_cur - _tgt).max()
print("Worst per-stratum deviation from the 70/15/15 target (rows):", round(_dev, 1))
assert _dev <= 4, "Split is not balanced across Category x Severity"

# 3)leakage checks
for _a, _b in [("train", "val"), ("train", "test"), ("val", "test")]:
    _ma, _mb = _split == _a, _split == _b
    assert not set(_text[_ma]) & set(_text[_mb]), f"Exact text overlap {_a}/{_b}"
    assert not set(_key[_ma]) & set(_key[_mb]), f"Normalized text overlap {_a}/{_b}"
    assert not set(_grp[_ma]) & set(_grp[_mb]), f"Near-duplicate group shared by {_a}/{_b}"
assert all(_split[i] == _split[j] for i, j in _links), "A linked pair crosses splits"
assert len(set(_strata)) == len(set(_strata[_split == "train"])) == len(set(_strata[_split == "val"])) == len(set(_strata[_split == "test"]))

# 4)task-specific frames
train_df = data[df["Split"] == "train"].reset_index(drop=True)
valid_df = data[df["Split"] == "val"].reset_index(drop=True)
test_df  = data[df["Split"] == "test"].reset_index(drop=True)
assert len(train_df) + len(valid_df) + len(test_df) == len(data)
assert set(train_df["label"]) == set(valid_df["label"]) == set(test_df["label"])

print("Linked pairs:", len(_links), "| groups with >1 row:", int((np.bincount(_grp) > 1).sum()))
print("Train:", train_df.shape, "| Validation:", valid_df.shape, "| Test:", test_df.shape)
print("Split fingerprint:", hashlib.sha1(",".join(map(str, sorted(df.loc[df["Split"] == "test", "No."]))).encode()).hexdigest()[:12],
      "(must be identical in all 6 notebooks)")
print("NO LEAKAGE: exact text, normalized text and near-duplicate groups are disjoint across splits")

Worst per-stratum deviation from the 70/15/15 target (rows): 0.8
Linked pairs: 94 | groups with >1 row: 68
Train: (4176, 2) | Validation: (899, 2) | Test: (895, 2)
Split fingerprint: 9dc9e17dd435 (must be identical in all 6 notebooks)
NO LEAKAGE: exact text, normalized text and near-duplicate groups are disjoint across splits


In [7]:
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

In [8]:
model_name = "csebuetnlp/banglabert"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [9]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )


train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/4176 [00:00<?, ? examples/s]

Map:   0%|          | 0/899 [00:00<?, ? examples/s]

Map:   0%|          | 0/895 [00:00<?, ? examples/s]

In [10]:
train_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

valid_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

test_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

In [11]:
set_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoi

In [12]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),

        "precision": precision_score(
            labels,
            predictions,
            average="macro"
        ),

        "recall": recall_score(
            labels,
            predictions,
            average="macro"
        ),

        "f1": f1_score(
            labels,
            predictions,
            average="macro"
        )
    }

In [13]:
training_args = TrainingArguments(
    output_dir="./banglabert_severity",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
    seed=42,
    data_seed=42
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [15]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,2.148877,2.116331,0.430478,0.304480,0.372753,0.301188
2,1.986967,2.134885,0.433815,0.489208,0.400996,0.362302
3,1.670109,2.263522,0.451613,0.463756,0.422429,0.418315
4,1.395773,2.576420,0.433815,0.450385,0.418082,0.409302
5,1.154734,2.600151,0.457175,0.465178,0.429226,0.421783


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=1305, training_loss=1.6826287996723277, metrics={'train_runtime': 414.801, 'train_samples_per_second': 50.337, 'train_steps_per_second': 3.146, 'total_flos': 1373452040540160.0, 'train_loss': 1.6826287996723277, 'epoch': 5.0})

In [16]:
result = trainer.evaluate(test_dataset)

result

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 2.692915916442871,
 'eval_accuracy': 0.4201117318435754,
 'eval_precision': 0.4256120201209937,
 'eval_recall': 0.39256046770802727,
 'eval_f1': 0.3876206686087467,
 'eval_runtime': 5.3242,
 'eval_samples_per_second': 168.1,
 'eval_steps_per_second': 10.518,
 'epoch': 5.0}

In [17]:
pred = trainer.predict(test_dataset)

y_pred = np.argmax(pred.predictions, axis=1)

y_true = pred.label_ids
assert (y_true == test_df["label"].values).all()

print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

        High       0.43      0.20      0.28       242
         Low       0.43      0.36      0.39       292
      Medium       0.41      0.61      0.49       361

    accuracy                           0.42       895
   macro avg       0.43      0.39      0.39       895
weighted avg       0.42      0.42      0.40       895



In [18]:
model_save_path = "./BanglaBERT_Severity_Model"

trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print("Severity model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Severity model saved!


In [19]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred, labels=range(num_labels))
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{c}" for c in label_encoder.classes_],
    columns=[f"pred_{c}" for c in label_encoder.classes_]
)
print("Confusion Matrix:")
print(cm_df)

Confusion Matrix:
             pred_High  pred_Low  pred_Medium
true_High           49        50          143
true_Low            14       106          172
true_Medium         50        90          221
